# This is where we can train our own model

In [139]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score

Load the data for training

In [130]:
df = pd.read_csv("../data/train_images.csv")

In [131]:
class BirdDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

        # Convert labels to numeric if needed
        self.classes = sorted(self.df['label'].unique())
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open('../data' + row["image_path"]).convert("RGB")

        if self.transform:
            img = self.transform(img)

        label = self.class_to_idx[row["label"]]
        return img, label


In [132]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

In [133]:
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

test_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])


train_dataset = BirdDataset(train_df, transform=train_tfms)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [134]:
classes = train_df['label'].unique()
classes

array([ 42, 193,  23,  49,  85,   5,  69, 179,  60,  56,  20,  12,  67,
        65,  84,  70, 186,  38,  21, 124,  17,  86, 147,  33,  77,  68,
       187,   6,  87,  61,  30, 199,  99,  31,  94,  45, 136, 146, 118,
        72,  14, 113,  59, 108, 126,   1, 200, 181, 166,  53, 185,  22,
        62,  16,  48, 172,  15, 125,  75, 140, 153, 133, 103,   9, 112,
       127, 131,  95,  13, 145,   7, 139, 101,  78,  88, 100,  11,  32,
        10,  35,  50, 169,  34, 178, 157, 142,  76,  58,  63,  93,  52,
         2,   3,  74, 110, 141,  64,  29, 120,  46,  97,  83,  57,  39,
        36,  40, 190, 188,  80, 130, 128, 105, 107,  71, 135,  91, 159,
        92,   4, 161, 163, 122,  47,   8,  43,  96,  25, 170,  79, 175,
        73,  19, 121, 149, 137,  44, 150,  55, 111, 194, 114,  90, 115,
        51,  89, 154, 152, 148, 155, 104, 123,  27, 109, 134,  24,  81,
       177,  26, 174, 164,  41,  54, 167, 184,  28, 158, 132, 119, 156,
       195,  66, 162,  18, 173, 117, 182, 160,  37, 191, 143, 19

Train the model

In [160]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5)
        self.fc1 = nn.Linear(16 * 53 * 53, 400)
        self.fc2 = nn.Linear(400, 300)
        self.fc3 = nn.Linear(300, len(classes))

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

net = Net()


In [161]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.0003)

In [162]:
for epoch in range(30):  # loop over the dataset multiple times
    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()

    print(f'[{epoch + 1}] loss: {running_loss/len(train_loader):.3f}')
    running_loss = 0.0

print('Finished Training')

[1] loss: 5.259
[2] loss: 5.107
[3] loss: 4.951
[4] loss: 4.695
[5] loss: 4.422
[6] loss: 4.201
[7] loss: 3.986
[8] loss: 3.793
[9] loss: 3.566
[10] loss: 3.368
[11] loss: 3.137
[12] loss: 2.903
[13] loss: 2.681
[14] loss: 2.433
[15] loss: 2.202
[16] loss: 1.983
[17] loss: 1.764
[18] loss: 1.530
[19] loss: 1.389
[20] loss: 1.228
[21] loss: 1.041
[22] loss: 0.850
[23] loss: 0.718
[24] loss: 0.617
[25] loss: 0.530
[26] loss: 0.422
[27] loss: 0.388
[28] loss: 0.341
[29] loss: 0.340
[30] loss: 0.234
Finished Training


Run the model on the val dataset

In [151]:
idx_to_class = {v: k for k, v in train_dataset.class_to_idx.items()}

In [163]:
val_dataset = BirdDataset(val_df, transform=test_tfms)
test_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

val_preds = []
val_labels = []
net.eval()

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = net(inputs)
        _, predicted = torch.max(outputs, 1)

        val_preds.extend(predicted.numpy())
        val_labels.extend(labels.numpy())


In [164]:
print(len(val_labels))

786


In [165]:
val_results_df = pd.DataFrame(
    columns=["actual", "predicted"],
    data={
        "actual": val_labels,
        "predicted": val_preds
    }
)

accuracy = accuracy_score(val_labels, val_preds)
print(accuracy)
val_results_df

0.07760814249363868


,actual,predicted
0,69,116
1,58,52
2,80,27
3,24,24
4,84,55
...,...,...
781,44,52
782,109,90
783,0,7
784,121,84


Run the model on test data

In [166]:
test_df = pd.read_csv("../data/test_images_path.csv")

In [167]:
test_dataset = BirdDataset(test_df, transform=test_tfms)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [168]:
net.eval()

Net(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=44944, out_features=400, bias=True)
  (fc2): Linear(in_features=400, out_features=300, bias=True)
  (fc3): Linear(in_features=300, out_features=200, bias=True)
)

In [169]:
all_ids = test_df["id"].tolist()
all_preds = []

In [170]:
with torch.no_grad():
    for inputs, _ in test_loader:
        outputs = net(inputs)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.numpy())

In [171]:
predicted_labels = [idx_to_class[i] for i in all_preds]

In [172]:
output_df = pd.DataFrame({
    "id": all_ids,
    "label": predicted_labels
})

output_df.to_csv("test_predictions.csv", index=False)
print("Saved test_predictions.csv!")

Saved test_predictions.csv!


In [53]:
len(train_df)

3926